# Phase Retrieval : PSF → WFE → Zernikes

Ce notebook fusionne, derrière une interface commune, les **trois méthodes**
de récupération de phase développées dans ce projet pour convertir une PSF (+ un
masque pupillaire binaire) en coefficients de Zernike (ordre de Noll) :

| Méthode | `method=` | Entrées | Principe |
|---|---|---|---|
| Gerchberg-Saxton | `"gerchberg_saxton"` | 1 PSF | Récupération de phase itérative par projections FFT successives, puis décomposition Zernike de la phase obtenue |
| Optimisation directe | `"optimization"` | 1 PSF | Ajustement des coefficients de Zernike par moindres carrés non-linéaires (`scipy.optimize.least_squares`) sur la PSF |
| Diversité de phase modale | `"phase_diversity"` | **2 PSF** (focus + défocus connu) | Ajustement conjoint des coefficients sur les deux PSF, ce qui lève l'ambiguïté de signe des deux méthodes précédentes |

**Tout le code numérique est externalisé** dans le paquet
`eoqual.phaseretrieval` (voir la section Imports) : le notebook ne fait
qu'appeler ce paquet et présenter/valider les résultats. Une section
**« Fonctionnalités avancées »**, plus loin, couvre le module de simulation
en mémoire, la pupille obstruée (`orthogonal_basis`) et la WFE haut ordre
connue (`known_ho_wfe_nm`) — non représentées par les données
`data/*_simu.tiff` utilisées jusque-là (pupille pleine, sans WFE haut ordre).

## Sur la fusion

Les méthodes Gerchberg-Saxton et optimisation partagent la même signature
(une seule PSF) : les fusionner derrière un simple sélecteur `method=` dans
[`pipeline.psf_to_wfe`](../src/eoqual/phaseretrieval/pipeline.py) est direct et évite la duplication de
tout le code commun (chargement PSF/masque, décomposition Zernike finale,
diagnostics).

La diversité de phase, elle, a besoin d'une **seconde** PSF et d'un défocus
connu : ce n'est pas une simple variante d'algorithme mais un protocole de
mesure différent. Elle **est** intégrée au même point d'entrée unifié
(`method="phase_diversity"`, avec `diversity_path=` et `defocus=`
supplémentaires) pour bénéficier des mêmes diagnostics et de la même
interface CLI, mais son cœur numérique reste dans son propre module
([`backends/phase_diversity.py`](../src/eoqual/phaseretrieval/backends/phase_diversity.py)) plutôt que
d'être mélangé aux méthodes mono-PSF.

## Versions abandonnées

- `gerchberg_saxton_V1` (déroulement de phase) → remplacé par la variante V2
  (plus stable), seule conservée ici sous le nom `gerchberg_saxton`.
- La version V1 de la diversité de phase modale → remplacée par la V2
  (pondération de bruit Poisson + ROI + perte robuste), seule intégrée ici.
- Les anciens notebooks/scripts monolithiques pré-fusion (un par méthode)
  ont été abandonnés au profit des modules ci-dessus.
- `simu_psf_mask.ipynb` (génération des données `data/*_simu.tiff`), à côté
  de ce notebook, reste hors fusion : c'est un outil de simulation, pas une
  méthode de récupération de phase — le module
  `eoqual.phaseretrieval.simulation` (section « Fonctionnalités avancées »)
  couvre désormais le même besoin sans fichier intermédiaire, pour les cas
  qui ne nécessitent pas la mosaïque Bayer multi-bande.

In [1]:
if not '__file__' in globals():  #HACK: pour éviter ces lignes non commentées quand on utilise manuellement nbconvert
    %load_ext autoreload
    %autoreload 2

## Imports

In [2]:
# Rend le paquet importable sans installation préalable
# (pip install -e . depuis la racine du dépôt fonctionne aussi).
import sys
from pathlib import Path as _Path
sys.path.insert(0, str((_Path.cwd() / ".." / "src").resolve()))

# ======================================================================
# IMPORTS STANDARD / TIERS / MODULES DU PROJET
# ======================================================================
# IMPORTS STANDARD
from pathlib import Path

# IMPORTS TIERS
import numpy as np
import plotly.graph_objects as go
from loguru import logger

# MODULES DU PROJET (voir les fichiers .py à côté de ce notebook)
from eoqual.phaseretrieval.io_utils import load_image, load_and_prepare_mask
from eoqual.phaseretrieval.zernike_fit import ZernikeConfig, fit_zernike
from eoqual.phaseretrieval.pipeline import psf_to_wfe, build_arg_parser, METHODS

logger.success("Fin des imports !")

2026-09-10 13:48:47.478 | SUCCESS  | __main__:<module>:23 - Fin des imports !


## Données du simulateur & paramètres communs

On utilise les données produites par `simu_psf_mask.ipynb` (`Data/*_simu.tiff`) pour valider les trois méthodes : la phase injectée (`phase_simu.tiff`) sert de vérité terrain, décomposée elle aussi sur la base de Zernike pour servir de référence de comparaison.

In [3]:
DATA_DIR = Path("data")

# Paramètres physiques communs aux trois méthodes
COMMON = dict(
    psf_path=DATA_DIR / "psf_simu.tiff",
    mask_path=DATA_DIR / "mask_simu.tiff",
    wavelength=550e-9,
    focal_length=4.0,
    pupil_diameter=0.5,
    pixel_pitch=4e-6,
    zernike_order=15,
    verbose=False,  # mettre à True pour les diagnostics Plotly interactifs
)

# Vérité terrain : décomposition Zernike de la phase injectée par le simulateur
mask_ref, _ = load_and_prepare_mask(
    psf_shape=(256, 256),
    mask_path=COMMON["mask_path"],
    wavelength=COMMON["wavelength"],
    focal_length=COMMON["focal_length"],
    pupil_diameter=COMMON["pupil_diameter"],
    pixel_pitch=COMMON["pixel_pitch"],
)
phase_true_rad = load_image(DATA_DIR / "phase_simu.tiff")
coeffs_true_nm, _, _ = fit_zernike(
    phase_true_rad, mask_ref, ZernikeConfig(order=COMMON["zernike_order"], wavelength=COMMON["wavelength"])
)

print("Coefficients de Zernike de la vérité terrain (nm WFE) :")
for i, c in enumerate(coeffs_true_nm, start=1):
    if abs(c) > 1e-6:
        print(f"   > Z{i:2d}: {c:8.2f} nm")

2026-09-10 13:48:47.512 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


Coefficients de Zernike de la vérité terrain (nm WFE) :
   > Z 4:    55.00 nm
   > Z 5:    27.50 nm


## Méthode 1 — Gerchberg-Saxton (`method="gerchberg_saxton"`, 1 PSF)

Récupération de phase itérative par projections successives entre plan pupille et plan focal. L'initialisation de la phase pupillaire est **aléatoire** (`seed=` pour la reproductibilité) : l'algorithme converge parfois vers la phase correcte, parfois vers son opposé (ambiguïté de signe), et peut aussi stagner selon le tirage initial (voir la section Validation).

In [4]:
coeffs_gs, phase_fit_gs_nm, diag_gs = psf_to_wfe(
    method="gerchberg_saxton", max_iter=300, seed=2, **COMMON,
)

print("Coefficients Gerchberg-Saxton (nm WFE) :")
for i, c in enumerate(coeffs_gs[:8], start=1):
    print(f"   > Z{i:2d}: {c:8.2f} nm")
print("Diagnostics :", diag_gs)

2026-09-10 13:48:47.572 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


Coefficients Gerchberg-Saxton (nm WFE) :
   > Z 1:     0.00 nm
   > Z 2:     0.00 nm
   > Z 3:     0.00 nm
   > Z 4:    24.76 nm
   > Z 5:    18.55 nm
   > Z 6:     1.35 nm
   > Z 7:     0.26 nm
   > Z 8:     0.78 nm
Diagnostics : {'method': 'gerchberg_saxton', 'matrix_rank': 15, 'condition_number': 6.579072824709474, 'phase_rms_nm': 37.39531975999509, 'fit_rms_nm': 36.450474817883936, 'residual_rms_nm': 117.14819622126689, 'residual_std_nm': 8.344220514323403}


## Méthode 2 — Optimisation directe (`method="optimization"`, 1 PSF)

Les coefficients de Zernike sont ajustés directement pour que la PSF simulée reproduise la PSF mesurée (`scipy.optimize.least_squares`). Plus rapide et déterministe que Gerchberg-Saxton (pas d'initialisation aléatoire), mais **souffre de la même ambiguïté de signe** : rien dans une PSF unique ne permet de distinguer une aberration de son opposée.

In [5]:
coeffs_opt, phase_fit_opt_nm, diag_opt = psf_to_wfe(
    method="optimization", fit_piston_tilt=False, **COMMON,
)

print("Coefficients Optimisation directe (nm WFE) :")
for i, c in enumerate(coeffs_opt[:8], start=1):
    print(f"   > Z{i:2d}: {c:8.2f} nm")
print("Diagnostics :", diag_opt)

2026-09-10 13:48:48.318 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


Coefficients Optimisation directe (nm WFE) :
   > Z 1:     0.00 nm
   > Z 2:     0.00 nm
   > Z 3:     0.00 nm
   > Z 4:   -54.61 nm
   > Z 5:   -27.20 nm
   > Z 6:     0.00 nm
   > Z 7:    -1.25 nm
   > Z 8:    -1.25 nm
Diagnostics : {'method': 'optimization', 'cost': 6.390760779193004e-05, 'nfev': 17, 'status': 1, 'message': '`gtol` termination condition is satisfied.', 'psf_rmse': 7.4985309612235534e-06}


## Méthode 3 — Diversité de phase modale (`method="phase_diversity"`, 2 PSF)

Une seconde PSF, acquise avec un défocus connu (`defocus=`), est ajustée **conjointement** avec la PSF focalisée. C'est la seule des trois méthodes qui nécessite `diversity_path=` — l'information supplémentaire apportée par le second cliché lève l'ambiguïté de signe des deux méthodes précédentes.

In [6]:
coeffs_pd, phase_fit_pd_nm, diag_pd = psf_to_wfe(
    method="phase_diversity",
    diversity_path=DATA_DIR / "psf_diversity_simu.tiff",
    defocus=275e-9,
    roi_radius=32,
    **COMMON,
)

print("Coefficients Diversité de phase (nm WFE) :")
for i, c in enumerate(coeffs_pd[:8], start=1):
    print(f"   > Z{i:2d}: {c:8.2f} nm")
print("Diagnostics :", diag_pd)

2026-09-10 13:48:49.104 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


Coefficients Diversité de phase (nm WFE) :
   > Z 1:     0.00 nm
   > Z 2:     0.00 nm
   > Z 3:     0.00 nm
   > Z 4:    55.00 nm
   > Z 5:    27.50 nm
   > Z 6:    -0.00 nm
   > Z 7:     0.00 nm
   > Z 8:     0.00 nm
Diagnostics : {'method': 'phase_diversity', 'cost': 0.0, 'nfev': 6, 'status': 1, 'message': '`gtol` termination condition is satisfied.', 'psf_focus_rmse': 9.730489434522509e-12, 'psf_defocus_rmse': 8.212792467707613e-12}


## Validation croisée sur les données du simulateur

On compare les coefficients de Zernike obtenus par chaque méthode à la vérité terrain (`coeffs_true_nm`, décomposition directe de la phase injectée par le simulateur). Gerchberg-Saxton étant sensible à son initialisation aléatoire, on le rejoue sur plusieurs graines pour quantifier sa variabilité plutôt que de ne montrer qu'un seul tirage.

In [7]:
def rms_error(coeffs_nm: np.ndarray) -> float:
    """RMS error [nm] against ground truth, over modes 4..zernike_order."""
    return float(np.sqrt(np.mean((coeffs_nm[3:] - coeffs_true_nm[3:]) ** 2)))


def correlation(coeffs_nm: np.ndarray) -> float:
    """Pearson correlation with ground truth, over modes 4..zernike_order."""
    return float(np.corrcoef(coeffs_true_nm[3:], coeffs_nm[3:])[0, 1])


# --- Robustesse de Gerchberg-Saxton sur plusieurs initialisations ---
gs_seeds = range(8)
gs_runs = {
    seed: psf_to_wfe(method="gerchberg_saxton", max_iter=300, seed=seed, **COMMON)[0]
    for seed in gs_seeds
}

print(f"{'seed':>6} {'RMS (nm)':>10} {'corr':>8}")
for seed, c in gs_runs.items():
    print(f"{seed:6d} {rms_error(c):10.2f} {correlation(c):8.3f}")

best_seed = min(gs_runs, key=lambda s: rms_error(gs_runs[s]))
coeffs_gs_best = gs_runs[best_seed]
print(f"\nMeilleure initialisation : seed={best_seed} "
      f"(RMS={rms_error(coeffs_gs_best):.2f} nm, corr={correlation(coeffs_gs_best):+.3f})")

2026-09-10 13:48:49.546 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


2026-09-10 13:48:50.387 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


2026-09-10 13:48:51.223 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


2026-09-10 13:48:52.068 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


2026-09-10 13:48:52.892 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


2026-09-10 13:48:53.724 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


2026-09-10 13:48:54.539 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


2026-09-10 13:48:55.332 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


  seed   RMS (nm)     corr
     0      21.41   -0.009
     1      28.39    0.337
     2      11.32    0.832
     3      34.67   -0.999
     4      23.55   -0.395
     5      21.18   -0.009
     6      41.09    0.594
     7     114.62   -0.523

Meilleure initialisation : seed=2 (RMS=11.32 nm, corr=+0.832)


In [8]:
results = {
    "Gerchberg-Saxton (best of 8)": coeffs_gs_best,
    "Optimisation": coeffs_opt,
    "Diversité de phase": coeffs_pd,
}

print(f"{'Méthode':<32} {'RMS (nm)':>10} {'corr':>8}")
for name, c in results.items():
    print(f"{name:<32} {rms_error(c):10.2f} {correlation(c):8.3f}")

# --- Comparaison graphique des premiers modes ---
n_show = 8
modes = np.arange(4, n_show + 1)
mode_labels = [f"Z{m}" for m in modes]

fig = go.Figure()
fig.add_bar(x=mode_labels, y=coeffs_true_nm[3:n_show], name="Vérité terrain")
for name, c in results.items():
    fig.add_bar(x=mode_labels, y=c[3:n_show], name=name)
fig.update_layout(
    barmode="group",
    title="Comparaison des coefficients de Zernike récupérés (données simulées)",
    yaxis_title="Coefficient [nm WFE]",
    width=750, height=420,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.add_hline(y=0, line_color="black", line_width=0.8)
fig.show()

Méthode                            RMS (nm)     corr
Gerchberg-Saxton (best of 8)          11.32    0.832
Optimisation                          35.37   -1.000
Diversité de phase                     0.00    1.000


### Interprétation

- **Gerchberg-Saxton** est le plus instable des trois : son initialisation aléatoire fait varier fortement la qualité de la reconstruction d'un tirage à l'autre (certaines graines stagnent, d'autres récupèrent la phase quasi correcte **ou son opposée**).
- **L'optimisation directe** est déterministe et récupère systématiquement la bonne *amplitude* des aberrations, mais avec une corrélation proche de **-1** avec la vérité : c'est l'ambiguïté de signe fondamentale de la récupération de phase à partir d'une seule PSF (une aberration et son opposé produisent la même image en intensité).
- **La diversité de phase** est la seule méthode qui lève cette ambiguïté grâce à la seconde PSF défocalisée : sur ces données simulées (sans bruit), elle récupère la phase **quasi exactement** (RMS ≈ 0, corrélation ≈ +1).

Ce constat confirme, a posteriori, le choix architectural : la diversité de phase n'est pas interchangeable avec les deux autres méthodes — elle répond à une limitation physique qu'elles ne peuvent pas lever, d'où sa conservation en tant que méthode à part entière (avec sa propre entrée de données) plutôt qu'une simple variante d'algorithme.

## Lever l'ambiguïté de signe : a priori sur un mode pair (Z13 / Z15)

Comme détaillé dans [`REFERENCE_TECHNIQUE.md` §3.7](REFERENCE_TECHNIQUE.md#37-lever-lambiguïté-de-signe-par-un-a-priori),
l'ambiguïté « image jumelle » d'une PSF unique n'affecte que les modes de
**fréquence azimutale paire** ($Z_4$ défocus, $Z_5$/$Z_6$ astigmatisme,
$Z_{11}$ sphérique, $Z_{12}$/$Z_{13}$ astigmatisme secondaire, …) — et ils
basculent **tous ensemble**. Les modes de fréquence **impaire** (coma
$Z_7$/$Z_8$, trèfle $Z_9$/$Z_{10}$, …) sont récupérés avec le bon signe
**dès une seule PSF**, sans aucune ambiguïté.

On le vérifie ici sur une phase synthétique multi-modes (coefficients
aléatoires sur $Z_4 \dots Z_N$, $N=13$ puis $N=15$), en utilisant le vrai
masque du simulateur et les mêmes briques que le pipeline (`optics.py` pour
générer la PSF de référence à partir de coefficients connus,
`backends.optimization` / `backends.gerchberg_saxton` + `zernike_fit` pour
la récupérer).

In [9]:
import poppy.zernike as zern
from eoqual.phaseretrieval.optics import build_zernike_basis, phase_from_coefficients, psf_from_phase
from eoqual.phaseretrieval.backends.gerchberg_saxton import RetrievalConfig, gerchberg_saxton
from eoqual.phaseretrieval.backends.optimization import OpticalConfig, fit_zernike_from_psf
from eoqual.phaseretrieval.zernike_fit import even_azimuthal_mask, resolve_sign_ambiguity

RAD_TO_NM = COMMON["wavelength"] * 1e9 / (2 * np.pi)


def synthetic_case(order: int, seed: int):
    """Build a synthetic multi-mode phase/PSF pair and its true Zernike coefficients."""
    rng = np.random.default_rng(seed)
    coeffs_nm = np.zeros(order)
    coeffs_nm[3:] = rng.uniform(-60, 60, order - 3)
    basis = build_zernike_basis(npix=256, nterms=order)
    phase_rad = phase_from_coefficients(coeffs_nm / RAD_TO_NM, basis, mask_ref)
    psf_synth = psf_from_phase(phase_rad, mask_ref)
    return coeffs_nm, psf_synth


def corr_rms(a: np.ndarray, b: np.ndarray) -> tuple:
    return float(np.corrcoef(a[3:], b[3:])[0, 1]), float(np.sqrt(np.mean((a[3:] - b[3:]) ** 2)))

In [10]:
# --- Cas de démonstration : Z4..Z15, un seul tirage ---
ORDER_DEMO = 15
true_demo_nm, psf_demo = synthetic_case(ORDER_DEMO, seed=2)

opt_cfg = OpticalConfig(wavelength=COMMON["wavelength"], zernike_order=ORDER_DEMO, verbose=False)
coeffs_raw, _ = fit_zernike_from_psf(psf_demo, mask_ref, opt_cfg)
coeffs_fixed = resolve_sign_ambiguity(coeffs_raw, reference_index=4, expected_sign=np.sign(true_demo_nm[3]))

even = even_azimuthal_mask(ORDER_DEMO)
print(f"{'j':>3} {'m parity':>9} {'true (nm)':>10} {'raw (nm)':>10} {'corrected (nm)':>15}")
for j in range(4, ORDER_DEMO + 1):
    parity = "even" if even[j-1] else "odd"
    print(f"{j:3d} {parity:>9} {true_demo_nm[j-1]:10.2f} {coeffs_raw[j-1]:10.2f} {coeffs_fixed[j-1]:15.2f}")

  j  m parity  true (nm)   raw (nm)  corrected (nm)
  4      even     -28.61      29.42          -29.42
  5      even     -24.18      23.68          -23.68
  6      even      37.71     -38.79           38.79
  7       odd     -48.97     -48.16          -48.16
  8       odd      12.01      10.89           10.89
  9       odd      27.43      26.33           26.33
 10       odd     -37.45     -39.53          -39.53
 11      even     -53.38      53.33          -53.33
 12      even     -27.00      26.97          -26.97
 13      even      18.89     -18.98           18.98
 14      even       7.47      -7.74            7.74
 15      even     -41.99      41.76          -41.76


Les modes impairs (coma, trèfle, …) ont déjà le bon signe avant même correction ; seuls les modes pairs étaient inversés, et la correction (a priori sur le seul $Z_4$) les remet tous d'aplomb simultanément.

In [11]:
# --- Robustesse statistique : 10 tirages aléatoires, ordre 13 et 15 ---
for order in (13, 15):
    print(f"\n=== zernike_order = {order} ===")
    n_ok = 0
    opt_cfg = OpticalConfig(wavelength=COMMON["wavelength"], zernike_order=order, verbose=False)
    for trial in range(10):
        true_nm, psf_synth = synthetic_case(order, seed=trial)
        coeffs_raw, diag = fit_zernike_from_psf(psf_synth, mask_ref, opt_cfg)
        coeffs_fixed = resolve_sign_ambiguity(coeffs_raw, 4, np.sign(true_nm[3]))
        corr, rms = corr_rms(coeffs_fixed, true_nm)
        ok = corr > 0.99
        n_ok += ok
        print(f"  trial={trial}  cost={diag['cost']:.1e}  corr={corr:+.3f}  RMS={rms:6.2f} nm  "
              f"{'OK' if ok else 'minimum local (nécessite un multi-start)'}")
    print(f"  -> {n_ok}/10 tirages correctement recalés après correction de signe")


=== zernike_order = 13 ===


  trial=0  cost=3.3e-02  corr=+0.359  RMS= 46.28 nm  minimum local (nécessite un multi-start)


  trial=1  cost=8.0e-29  corr=+1.000  RMS=  0.00 nm  OK


  trial=2  cost=9.8e-05  corr=+1.000  RMS=  0.81 nm  OK


  trial=3  cost=2.4e-04  corr=+1.000  RMS=  1.10 nm  OK


  trial=4  cost=6.4e-05  corr=+0.999  RMS=  1.38 nm  OK


... (11 outputs left out)

  trial=6  cost=4.8e-05  corr=+0.999  RMS=  1.46 nm  OK


  trial=7  cost=2.2e-27  corr=+1.000  RMS=  0.00 nm  OK


  trial=8  cost=1.7e-25  corr=+1.000  RMS=  0.00 nm  OK


  trial=9  cost=6.6e-05  corr=+0.999  RMS=  1.71 nm  OK
  -> 9/10 tirages correctement recalés après correction de signe


Les tirages non résolus au premier essai sont dus à une non-convergence générique du solveur (minimum local, `cost` non négligeable) — sans lien avec l'ambiguïté de signe — et se corrigent avec quelques redémarrages aléatoires de `least_squares` (voir §3.7 du document de référence).

In [12]:
# --- Même test avec Gerchberg-Saxton : la correction y est peu fiable ---
ORDER_GS = 15
true_nm, psf_synth = synthetic_case(ORDER_GS, seed=7)
zc = ZernikeConfig(order=ORDER_GS, wavelength=COMMON["wavelength"])

print(f"{'seed':>5} {'corr (brut)':>12} {'corr (corrigé)':>15}")
for seed in range(6):
    gs_cfg = RetrievalConfig(max_iter=300, seed=seed, verbose=False)
    phase_rad = gerchberg_saxton(psf_synth, mask_ref, gs_cfg)
    coeffs_raw, _, _ = fit_zernike(phase_rad, mask_ref, zc)
    coeffs_fixed = resolve_sign_ambiguity(coeffs_raw, 4, np.sign(true_nm[3]))

    corr_raw, _ = corr_rms(coeffs_raw, true_nm)
    corr_fixed, _ = corr_rms(coeffs_fixed, true_nm)
    print(f"{seed:5d} {corr_raw:+12.3f} {corr_fixed:+15.3f}")

 seed  corr (brut)  corr (corrigé)


    0       +0.778          +0.261


    1       +0.230          +0.230


    2       +0.076          +0.886


    3       +0.229          +0.229


    4       -0.278          +0.040


    5       +0.015          +0.999


Contrairement à l'optimisation directe, la correction n'améliore pas systématiquement Gerchberg-Saxton : quand celui-ci n'a pas convergé proprement vers l'une des deux branches exactes, le signe du mode de référence ($Z_4$) utilisé pour décider de la correction est lui-même peu fiable.

## Interface CLI unifiée

Les trois méthodes partagent le même point d'entrée en ligne de commande (`python pipeline.py ...`), sélectionné par `--method`.

In [13]:
print(build_arg_parser().format_help())

usage: ipykernel_launcher.py [-h] --psf PSF --mask MASK --output OUTPUT
                             [--method {gerchberg_saxton,optimization,phase_diversity}]
                             [--wavelength WAVELENGTH]
                             [--focal_length FOCAL_LENGTH]
                             [--pupil_diameter PUPIL_DIAMETER]
                             [--pixel_pitch PIXEL_PITCH]
                             [--oversampling OVERSAMPLING]
                             [--zernike_order ZERNIKE_ORDER]
                             [--mask_no_binarize]
                             [--mask_native_aperture_px MASK_NATIVE_APERTURE_PX]
                             [--ho_native_aperture_px HO_NATIVE_APERTURE_PX]
                             [--mask_native_aperture_center ROW COL]
                             [--ho_native_aperture_center ROW COL]
                             [--aperture_threshold APERTURE_THRESHOLD]
                             [--orthogonal_basis] [--max_iter MAX_ITER]

Exemples d'invocation :

```bash
# Gerchberg-Saxton
python pipeline.py --method gerchberg_saxton \
    --psf Data/psf_simu.tiff --mask Data/mask_simu.tiff --output Output \
    --zernike_order 15 --max_iter 300 --seed 2

# Optimisation directe
python pipeline.py --method optimization \
    --psf Data/psf_simu.tiff --mask Data/mask_simu.tiff --output Output \
    --zernike_order 15

# Diversité de phase (2 PSF)
python pipeline.py --method phase_diversity \
    --psf Data/psf_simu.tiff --diversity Data/psf_diversity_simu.tiff \
    --mask Data/mask_simu.tiff --output Output \
    --zernike_order 15 --defocus 275e-9 --roi_radius 32
```

## Visualisation interactive (Plotly), à la demande

`psf_to_wfe(..., verbose=True)` affiche la PSF mesurée, le masque, la carte de phase (mesurée le cas échéant, et reconstruite à partir du fit Zernike) ainsi que la PSF prédite. Exemple avec la diversité de phase (la méthode la plus fidèle sur ces données).

In [14]:
_ = psf_to_wfe(
    method="phase_diversity",
    diversity_path=DATA_DIR / "psf_diversity_simu.tiff",
    defocus=275e-9,
    roi_radius=32,
    **{**COMMON, "verbose": True},
)

2026-09-10 13:49:16.212 | WARNING  | eoqual.phaseretrieval.io_utils:load_and_prepare_mask:434 - Masque déjà sur la grille de la PSF (256x256) : mise à l'échelle physique ignorée — vérifier que c'est bien voulu si le masque vient d'un autre outil (voir REFERENCE_TECHNIQUE.md §12).


   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         2.3484e-01                                    9.66e-01    
       1              2         3.7323e-02      1.98e-01       8.76e-01       2.43e-01    


       2              3         6.1676e-04      3.67e-02       2.73e-01       2.97e-02    


       3              4         6.3562e-07      6.16e-04       5.49e-02       3.65e-04    
       4              5         3.7113e-12      6.36e-07       2.49e-03       9.40e-07    


       5              6         0.0000e+00      3.71e-12       6.05e-06       6.28e-12    
`gtol` termination condition is satisfied.
Function evaluations 6, initial cost 2.3484e-01, final cost 0.0000e+00, first-order optimality 6.28e-12.


## Fonctionnalités avancées

Les sections ci-dessous couvrent les évolutions du projet **non
représentées** par les données `data/*_simu.tiff` utilisées jusqu'ici
(toutes générées avec une pupille pleine non obstruée, sans WFE haut
ordre au-delà de `zernike_order`) : le module de simulation en mémoire
(`eoqual.phaseretrieval.simulation`, sans fichier à préparer), l'option
`orthogonal_basis` pour une pupille obstruée et son retour à la
nomenclature standard (`project_orthogonal_to_standard`), et
l'incorporation d'une carte de WFE haut ordre connue
(`known_ho_wfe_nm`), et les a priori individuels par mode
(`zernike_priors`/`zernike_bounds`). Voir `REFERENCE_TECHNIQUE.md`
§6.7, §10, §11.3 et §6.5bis pour le détail théorique de chacune.

### Module de simulation : cas de test sans fichier externe

Contrairement aux sections précédentes (PSF chargées depuis
`data/*.tiff`), `eoqual.phaseretrieval.simulation` génère une PSF (ou une
paire focus/défocus, avec bruit détecteur réaliste optionnel) directement
en mémoire, à partir de coefficients de Zernike connus — pratique pour un
test rapide sans préparer de fichiers.

Point d'architecture à noter : la simulation travaille avec les fonctions
**bas niveau** de chaque backend (`fit_zernike_from_psf`,
`fit_phase_diversity`, `gerchberg_saxton` + `fit_zernike`), pas avec
`psf_to_wfe` (orienté fichiers, utilisé dans tout ce notebook jusqu'ici) :
ce sont deux points d'entrée complémentaires, pas concurrents — voir
`REFERENCE_TECHNIQUE.md` §6.2 (point d'entrée unifié) vs §6.7 (module de
simulation).

In [15]:
from eoqual.phaseretrieval.simulation import (
    simulate_psf_from_zernike,
    simulate_phase_diversity_pair,
    PhaseDiversitySimulationConfig,
    DetectorNoiseConfig,
)
from eoqual.phaseretrieval.backends.optimization import OpticalConfig, fit_zernike_from_psf
from eoqual.phaseretrieval.backends.phase_diversity import PhaseDiversityConfig, fit_phase_diversity

ADV_WAVELENGTH = 550e-9
ADV_NPIX = 128
ADV_ORDER = 12  # ordre 15 sujet à un minimum local isolé pour "optimization" à ce npix
ADV_TRUTH = {4: 55.0, 5: 27.5}  # même vérité que data/*_simu.tiff, en mémoire cette fois

# Une PSF simulée avec bruit détecteur réaliste (Poisson + lecture), méthode "optimization"
sim = simulate_psf_from_zernike(
    ADV_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=ADV_ORDER,
    noise=DetectorNoiseConfig(photon_count=1e6, read_noise_e=5.0, seed=0),
)
opt_cfg = OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False)
coeffs_sim, diag_sim = fit_zernike_from_psf(sim["psf"], sim["mask"], opt_cfg)

print("Vérité simulée        : Z4=%.1f  Z5=%.1f" % (sim["coeffs_nm"][3], sim["coeffs_nm"][4]))
print("Ajustement (avec bruit): Z4=%.2f  Z5=%.2f" % (coeffs_sim[3], coeffs_sim[4]))
print("  (signe parfois inversé sur 1 PSF bruitée : ambiguïté attendue, §3.7 — comparer |corrélation|)")

# Une paire focus/défocus bruitée, méthode "phase_diversity"
pd_sim_cfg = PhaseDiversitySimulationConfig(
    wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=ADV_ORDER, defocus=275e-9,
    noise=DetectorNoiseConfig(photon_count=1e6, read_noise_e=5.0, seed=0),
)
pair = simulate_phase_diversity_pair(ADV_TRUTH, pd_sim_cfg)
pd_cfg = PhaseDiversityConfig(
    wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, defocus=pair["defocus"],
    verbose=False, max_nfev=300,
)
coeffs_pair, diag_pair = fit_phase_diversity(pair["psf_focus"], pair["psf_defocus"], pair["mask"], pd_cfg)
print("Diversité de phase (avec bruit) : Z4=%.2f  Z5=%.2f" % (coeffs_pair[3], coeffs_pair[4]))

Vérité simulée        : Z4=55.0  Z5=27.5
Ajustement (avec bruit): Z4=-55.13  Z5=-28.00
  (signe parfois inversé sur 1 PSF bruitée : ambiguïté attendue, §3.7 — comparer |corrélation|)


Diversité de phase (avec bruit) : Z4=61.88  Z5=19.10


### Pupille obstruée : `orthogonal_basis` et retour à la base standard

La base de Zernike standard n'est orthogonale que sur un disque plein
(§10) : sur une pupille obstruée (obstruction centrale, bras d'araignée —
le cas courant en télédétection, §8.1), `orthogonal_basis=True` (sur
`OpticalConfig`/`PhaseDiversityConfig`, et côté simulation pour rester
cohérent — §10.1) bascule sur une base ré-orthogonalisée par Gram-Schmidt
sur le masque réel : meilleur conditionnement, mais les coefficients ne
portent plus le nom usuel (« défocus », « astigmatisme »…) au-delà des
tout premiers ordres. `zernike_fit.project_orthogonal_to_standard`
reconvertit après coup vers la nomenclature standard pour
l'interprétation (§10.2) — au prix d'un léger écart quand la vérité
d'origine n'est pas exactement définie dans cette même base standard
(mélange de piston, voir §10.2).

In [16]:
from eoqual.phaseretrieval.pupil import PupilGenerator
from eoqual.phaseretrieval.zernike_basis_orthogonal import resolve_zernike_basis
from eoqual.phaseretrieval.zernike_fit import project_orthogonal_to_standard

obstructed_mask = PupilGenerator(ADV_NPIX).annular(obscuration=0.3)  # 30 % d'obstruction centrale

# Vérité pensée en base standard (aberration physique connue), ajustée en base Gram-Schmidt
sim_obstr = simulate_psf_from_zernike(
    ADV_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=12, mask=obstructed_mask,
)
opt_cfg_gs = OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=12, verbose=False, orthogonal_basis=True)
coeffs_gs_basis, diag_gs_basis = fit_zernike_from_psf(sim_obstr["psf"], obstructed_mask, opt_cfg_gs)

# Retour à la base standard pour l'interprétation
basis_gs = resolve_zernike_basis(npix=ADV_NPIX, nterms=12, mask=obstructed_mask, orthogonalize=True)
coeffs_std, phase_std_nm, diag_reproj = project_orthogonal_to_standard(
    coeffs_gs_basis, basis_gs, obstructed_mask, wavelength=ADV_WAVELENGTH,
)

print("Vérité (base standard)              : Z4=55.0  Z5=27.5")
print("Ajustement Gram-Schmidt, reprojeté  : Z4=%.2f  Z5=%.2f" % (coeffs_std[3], coeffs_std[4]))
print("condition_number (base standard, pupille obstruée) : %.2f" % diag_reproj["condition_number"])

Vérité (base standard)              : Z4=55.0  Z5=27.5
Ajustement Gram-Schmidt, reprojeté  : Z4=55.03  Z5=27.46
condition_number (base standard, pupille obstruée) : 1.54


### WFE haut ordre connue (`known_ho_wfe_nm`)

Une carte de WFE mesurée indépendamment (interférogramme, as-built,
calibration sol), au-delà de `zernike_order`, biaise les coefficients bas
ordre si elle n'est pas prise en compte dans le modèle (§11.2 :
identifiabilité). `known_ho_wfe_nm` (`OpticalConfig`/`PhaseDiversityConfig`,
ou `psf_to_wfe(..., known_ho_wfe_path=...)`/`--known_ho_wfe` en CLI pour
une carte sur fichier) l'ajoute au modèle direct sans jamais l'ajuster —
§11.3. Démonstration avec un unique mode $Z_{21}$ connu de 40 nm, hors de
portée de `zernike_order=12` ci-dessous.

In [17]:
from eoqual.phaseretrieval.optics import build_zernike_basis, phase_from_coefficients

full_disk_mask = PupilGenerator(ADV_NPIX).simple_disk()
rad_to_nm_adv = ADV_WAVELENGTH * 1e9 / (2 * np.pi)
basis25 = build_zernike_basis(npix=ADV_NPIX, nterms=25)
ho_coeffs = np.zeros(25)
ho_coeffs[20] = 40.0  # Z21, nm — hors de portée de zernike_order=12 ci-dessous
ho_wfe_nm = phase_from_coefficients(ho_coeffs / rad_to_nm_adv, basis25, full_disk_mask) * rad_to_nm_adv

sim_ho = simulate_psf_from_zernike(
    ADV_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=12,
    mask=full_disk_mask, known_ho_wfe_nm=ho_wfe_nm,
)

coeffs_biased, diag_biased = fit_zernike_from_psf(
    sim_ho["psf"], full_disk_mask,
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=12, verbose=False),
)
coeffs_corrected, diag_corrected = fit_zernike_from_psf(
    sim_ho["psf"], full_disk_mask,
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=12, verbose=False, known_ho_wfe_nm=ho_wfe_nm),
)

print("Vérité                    : Z4=55.0  Z5=27.5")
print("Sans la carte HO connue   : Z4=%.2f  Z5=%.2f  (biaisé)" % (coeffs_biased[3], coeffs_biased[4]))
print("Avec la carte HO connue   : Z4=%.2f  Z5=%.2f" % (coeffs_corrected[3], coeffs_corrected[4]))

Vérité                    : Z4=55.0  Z5=27.5
Sans la carte HO connue   : Z4=44.73  Z5=41.51  (biaisé)
Avec la carte HO connue   : Z4=55.00  Z5=27.50


### A priori individuels par mode : `zernike_priors` / `zernike_bounds`

`OpticalConfig.l2_reg` (déjà vu plus haut) est un cas particulier d'un
mécanisme plus général : un a priori bayésien **individuel** par mode
(moyenne et poids propres à chaque coefficient) plutôt qu'un unique poids
vers zéro pour tous les modes — voir `REFERENCE_TECHNIQUE.md` §6.5bis
pour la dérivation complète (MAP gaussien = pénalité L2, MAP uniforme =
borne dure ; `zernike_priors` en est la forme souple, `zernike_bounds` la
forme dure). Démonstration ci-dessous : à l'ordre 12, sans a priori, la
reconstruction mono-PSF souffre de l'ambiguïté de signe des modes pairs
(§3.7 — $Z_4$, $Z_6$ inversés ; $Z_8$, impair, jamais ambigu) ; un a
priori individuel fort et correct sur les modes concernés la résout tout
en la débiaisant, **sans** passer par une seconde image de diversité.

**Attention à l'origine d'un a priori dans ce contexte (télédétection,
§8)** : une décroissance systématique de l'amplitude avec l'ordre radial
(courante en optique adaptative sol, issue de la statistique de turbulence
de Kolmogorov) n'est *pas*, en général, justifiée pour la WFE propre d'un
instrument spatial — voir la mise en garde de §6.5bis. Les valeurs
ci-dessous sont des a priori "connus" arbitraires à des fins de
démonstration, pas une telle hypothèse générique.

In [18]:
PRIOR_TRUTH = {4: 60.0, 6: 35.0, 8: -25.0}  # Z4, Z6 pairs (ambigus sur 1 PSF) ; Z8 impair (jamais ambigu)
sim_prior = simulate_psf_from_zernike(PRIOR_TRUTH, wavelength=ADV_WAVELENGTH, npix=ADV_NPIX, zernike_order=ADV_ORDER)

coeffs_noprior, _ = fit_zernike_from_psf(
    sim_prior["psf"], sim_prior["mask"],
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False),
)

# A priori individuel fort (petit écart-type) et correct sur les modes pairs concernés
priors = {4: (60.0, 2.0), 6: (35.0, 2.0)}
coeffs_prior, _ = fit_zernike_from_psf(
    sim_prior["psf"], sim_prior["mask"],
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False, zernike_priors=priors),
)

print("V\u00e9rit\u00e9                         : Z4=%.1f  Z6=%.1f  Z8=%.1f" % (PRIOR_TRUTH[4], PRIOR_TRUTH[6], PRIOR_TRUTH[8]))
print("Sans a priori (1 PSF)          : Z4=%.2f  Z6=%.2f  Z8=%.2f" % (coeffs_noprior[3], coeffs_noprior[5], coeffs_noprior[7]))
print("  (signe de Z4/Z6 parfois invers\u00e9 : ambig\u00fcit\u00e9 des modes pairs, \u00a73.7)")
print("Avec a priori individuel Z4,Z6 : Z4=%.2f  Z6=%.2f  Z8=%.2f" % (coeffs_prior[3], coeffs_prior[5], coeffs_prior[7]))

# Contrainte physique (borne dure) : Z4 jamais négatif dans ce montage
coeffs_bounded, _ = fit_zernike_from_psf(
    sim_prior["psf"], sim_prior["mask"],
    OpticalConfig(wavelength=ADV_WAVELENGTH, zernike_order=ADV_ORDER, verbose=False, zernike_bounds={4: (0.0, 500.0)}),
)
print("Avec borne Z4 in [0, 500] nm   : Z4=%.2f" % coeffs_bounded[3])

Vérité                         : Z4=60.0  Z6=35.0  Z8=-25.0
Sans a priori (1 PSF)          : Z4=-59.65  Z6=-35.92  Z8=-26.86
  (signe de Z4/Z6 parfois inversé : ambigüité des modes pairs, §3.7)
Avec a priori individuel Z4,Z6 : Z4=60.00  Z6=35.00  Z8=-25.00


Avec borne Z4 in [0, 500] nm   : Z4=60.00


## Organisation du code

```
src/eoqual/phaseretrieval/
    io_utils.py                    # chargement image/masque, diagnostics Plotly
    optics.py                      # base de Zernike, phase -> PSF (modèle direct)
    zernike_fit.py                 # décomposition d'une carte de phase (Noll) + project_orthogonal_to_standard
    pipeline.py                    # psf_to_wfe(method=...) + CLI
    config.py                      # registre METHODS_CONFIGS
    runner.py                      # list_methods()
    zernike_basis_orthogonal.py    # ZernikeGramSchmidt + resolve_zernike_basis (§10)
    pupil.py                       # pupilles disque/annulaire/araignées
    diagnostics.py                 # Strehl, WFE (Maréchal), MTF, ellipticité
    backends/
        gerchberg_saxton.py        # Gerchberg-Saxton                 (1 PSF)
        optimization.py            # optimisation directe              (1 PSF)
        phase_diversity.py         # diversité de phase modale         (2 PSF)
    simulation/                    # module de simulation (section "Fonctionnalités avancées")
        psf_simulator.py
        phase_diversity_simulator.py
        noise.py
examples/
    psf2zernike_demo.ipynb         # ce notebook
    simu_psf_mask.ipynb            # génération de data/*_simu.tiff (Bayer multi-bande)
```